In [ ]:
"""
Environment Setup Module.
This cell installs the necessary quantum computing libraries in the Google Colab environment.
It must be executed before running any other cell.
"""
!pip install qiskit[visualization] qiskit-aer qiskit-ibm-runtime matplotlib pylatexenc

import numpy as np
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram
from IPython.display import display

print("\n ENVIRONMENT SETUP COMPLETE! You can now proceed to the next cell.")

### Exercise 1: Building the Quantum Search (3 Qubits)

**Motivation:**
Grover's algorithm provides a quadratic speedup for unstructured search problems. For a 3-qubit system, the search space contains $N=8$ possible states. Classically, finding the correct state takes an average of 4 checks. Quantumly, we can find it with high probability using only $k=2$ queries to the Oracle.

**Theoretical Context:**
The algorithm operates by alternating two phases:
1. **The Oracle:** Marks the target state (we will look for `111`) by flipping its phase to negative.
2. **The Diffuser:** Inverts all amplitudes around their mean. Because the target was made negative, this reflection massively amplifies its probability while shrinking the incorrect states.

**Your Task:**
1. Run the cell below with `NUM_ITERATIONS = 2`. This is the mathematical optimum for $N=8$.
2. **Observe:** The probability of measuring `111` will be roughly $94.5\%$.
3. **The Experiment (Over-rotation):** In classical algorithms, checking more times never hurts. In quantum mechanics, rotating too far pushes you away from the target! Change `NUM_ITERATIONS` to `1` (probability drops to ~78%) and then to `3` (probability drops to ~33%!). This proves that Grover's algorithm behaves like a geometric rotation, not a classical filter.

In [ ]:
"""
Grover's Algorithm Module.
Implements a 3-qubit search for the state |111>. Demonstrates the importance 
of the optimal iteration count (k*) to prevent over-rotation.
"""

# --- STUDENT EXPERIMENT ZONE ---
# 2 is the mathematical optimum. Try changing it to 1 or 3!
NUM_ITERATIONS = 2
# -------------------------------

def create_grover_circuit(iterations: int) -> QuantumCircuit:
    """
    Constructs a 3-qubit Grover circuit targeting the |111> state.
    
    Args:
        iterations (int): The number of times the Oracle+Diffuser block is applied.
        
    Returns:
        QuantumCircuit: The assembled quantum circuit ready for execution.
    """
    qc = QuantumCircuit(3, 3)

    # 1. INITIALIZATION: Uniform superposition
    qc.h([0, 1, 2])
    qc.barrier()

    for _ in range(iterations):
        # 2. THE ORACLE (Marks state |111> with a negative phase)
        # We use a Multi-Controlled Z (CCZ) gate, built via H-CCX-H
        qc.h(2)
        qc.ccx(0, 1, 2)
        qc.h(2)
        qc.barrier()

        # 3. THE DIFFUSER (Amplitude Amplification / Inversion about the mean)
        qc.h([0, 1, 2])
        qc.x([0, 1, 2])
        
        # CCZ logic
        qc.h(2)
        qc.ccx(0, 1, 2)
        qc.h(2)
        
        qc.x([0, 1, 2])
        qc.h([0, 1, 2])
        qc.barrier()

    # 4. MEASUREMENT
    qc.measure([0, 1, 2], [0, 1, 2])
    return qc

# Generate and display the circuit
grover_circ = create_grover_circuit(NUM_ITERATIONS)
display(grover_circ.draw('mpl'))

# Execute on local perfect simulator
simulator = AerSimulator()
job = simulator.run(grover_circ, shots=2000)
counts = job.result().get_counts()

print(f"\nResults after {NUM_ITERATIONS} iterations: {counts}")

# Analysis
target_shots = counts.get('111', 0)
accuracy = (target_shots / 2000) * 100
print(f" Probability of finding the target state |111>: {accuracy:.1f}%")

display(plot_histogram(counts, title=f"Grover Search (k={NUM_ITERATIONS})"))

### OPTIONAL: Execute on a Real Quantum Computer (QPU)

**The Reality of Quantum Hardware:**
Grover's algorithm is mathematically beautiful but physically demanding. It requires many multi-qubit gates (like the Toffoli `CCX`). On current NISQ (Noisy Intermediate-Scale Quantum) devices, every gate introduces a tiny bit of error, and qubits naturally lose their state over time (decoherence). 

By running our circuit on a real IBM Quantum chip, we will see how hardware noise degrades our perfect $94.5\%$ theoretical accuracy.

**Step 1: Get Your API Token & Check Quota**
1. Copy your API token from the [IBM Quantum Platform](https://quantum.ibm.com/).
2. Paste it into the `TOKEN` variable below and run the cell to authenticate.

In [ ]:
"""
IBM Quantum Authentication Module.
"""
from qiskit_ibm_runtime import QiskitRuntimeService

# --- STUDENT INPUT REQUIRED ---
# Replace with your personal IBM Quantum API Token
TOKEN = "YOUR_IBM_QUANTUM_TOKEN_HERE" 
# ------------------------------

def check_ibm_quota(api_token: str) -> None:
    if api_token == "YOUR_IBM_QUANTUM_TOKEN_HERE":
        print(" ACTION REQUIRED: Please insert your personal API token in the code above.")
        return

    print("Authenticating with IBM Quantum...")
    try:
        QiskitRuntimeService.save_account(channel="ibm_quantum", token=api_token, set_as_default=True, overwrite=True)
        service = QiskitRuntimeService()
        usage = service.usage()
        
        print("\n Authentication Successful!")
        print("=== Your IBM Quantum Compute Time ===")
        print(f"Usage Details: {usage}")
        print("=====================================")
    except Exception as e:
        print(f" Authentication Failed. Error: {e}")

check_ibm_quota(TOKEN)

**Step 2: Transpile and Execute on the QPU**
The code below uses the `SamplerV2` primitive. It will:
1. Find the least busy physical quantum computer.
2. **Transpile** our logical circuit (mapping our abstract qubits to physical superconducting circuits).
3. Send the job. *(Note: You may wait in a queue for several minutes).*

In [ ]:
"""
QPU Execution Module.
Executes Grover's circuit on real hardware and compares the noisy output 
against the theoretical expectation.
"""
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit_ibm_runtime import SamplerV2

def run_grover_on_hardware() -> None:
    try:
        service = QiskitRuntimeService()
    except Exception:
        print(" Please run the authentication cell above first.")
        return

    # 1. Select hardware
    print("Searching for the least busy quantum computer...")
    backend = service.least_busy(operational=True, simulator=False)
    print(f" Selected Backend: {backend.name} (Qubits: {backend.num_qubits})")

    # We must ensure we are using the optimal circuit (k=2) for the real test
    optimal_qc = create_grover_circuit(iterations=2)

    # 2. Transpile
    print("Transpiling logical circuit for specific hardware topology...")
    pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
    isa_circuit = pm.run(optimal_qc)

    # 3. Execute
    print(f"Sending job to {backend.name} queue... (Please wait)")
    sampler = SamplerV2(backend)
    job = sampler.run([isa_circuit])
    print(f"Job successfully submitted! Job ID: {job.job_id()}")
    
    # 4. Wait for results
    result = job.result()
    pub_result = result[0]
    counts = pub_result.data.c.get_counts()
    
    print(f"\n📡 Real Hardware Results Received: {counts}")
    
    # 5. Noise Analysis
    total_shots = sum(counts.values())
    target_shots = counts.get('111', 0)
    accuracy = (target_shots / total_shots) * 100
    
    print("\n=== NOISE ANALYSIS ===")
    print(f"Ideal Theoretical Output: ~94.5% '111'")
    print(f"Actual Hardware Accuracy: {accuracy:.2f}%")
    print("Notice how gate errors and decoherence flattened the distribution!")
    print("======================")
    
    display(plot_histogram(counts, title=f"Grover on Real QPU ({backend.name})"))

if TOKEN != "YOUR_IBM_QUANTUM_TOKEN_HERE":
    run_grover_on_hardware()